# Run ASTRID on patient p2 — notebook version

Notebook equivalent of the CLI run in `adrian_spec/test_run/`. Reuses the **same**
`p2_preASTRID.h5ad`, so results are directly comparable to the baseline.

**Kernel:** ASTRID (.venv310)  ·  **Runtime:** ~5 min  ·  **Writes:** ~3.6 GB

Do *not* use the cell-1 filter from `ASTRID_Vignette.ipynb` (`used_in_NSCLC_immune`) —
for p2 that keeps only 345 of 5,166 cells, so the run would not be comparable.


In [1]:
# --- setup ---------------------------------------------------------------
# ASTRID's R step (RunSingleR.R) loads its reference with RELATIVE paths
# ("data/ASTRID_SingleR_Reference_20240701.Rds"), so the working directory
# must be the repo root for the whole notebook.
import os, sys, subprocess, time
from pathlib import Path

REPO = Path("/Users/adriansohrabi/Documents/GitHub/astrid")
os.chdir(REPO)

import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (5, 5)
plt.rcParams["pdf.fonttype"] = 42
sns.set_theme(style="white")

assert (REPO / "data" / "ASTRID_SingleR_Reference_20240701.Rds").exists(), "wrong cwd"
print("cwd:", os.getcwd())
print("python:", sys.executable)
print("Rscript:", subprocess.run(["which", "Rscript"], capture_output=True, text=True).stdout.strip())

cwd: /Users/adriansohrabi/Documents/GitHub/astrid
python: /Users/adriansohrabi/Documents/GitHub/astrid/.venv310/bin/python
Rscript: /usr/local/bin/Rscript


/Users/adriansohrabi/Documents/GitHub/astrid/.venv310/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/Users/adriansohrabi/Documents/GitHub/astrid/.venv310/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/Users/adriansohrabi/Documents/GitHub/astrid/.venv310/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/Users/adriansohrabi/Documents/GitHub/astrid/.venv310/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx instead.
  warnings.warn(msg, FutureWarning)
/Users/adriansohrabi/Documents

In [2]:
# --- parameters ----------------------------------------------------------
sample      = "p2"
prefix      = sample
author_type = "Major cell type"          # the column in adata.obs with author labels

baseline    = REPO / "adrian_spec/test_run"       # the earlier CLI run — read only
outdir      = REPO / "adrian_spec/nb_run"         # this run writes here
outdir.mkdir(parents=True, exist_ok=True)

adata_file        = baseline / f"{sample}_preASTRID.h5ad"      # same input as the CLI run
output_file       = outdir   / f"{sample}_outASTRID.h5ad"
clustering_result = outdir   / f"{sample}_ASTRID_Result.csv"
log_file          = outdir   / "run.log"

assert adata_file.exists(), adata_file
print(f"in : {adata_file}  ({adata_file.stat().st_size/1e6:.0f} MB)")
print(f"out: {outdir}/")

in : /Users/adriansohrabi/Documents/GitHub/astrid/adrian_spec/test_run/p2_preASTRID.h5ad  (42 MB)
out: /Users/adriansohrabi/Documents/GitHub/astrid/adrian_spec/nb_run/


In [ ]:
# --- look at the input before running ------------------------------------
adata = sc.read_h5ad(adata_file)
print(adata)
print()
print(adata.obs[author_type].value_counts().head(10))
print()
print("tissue:", dict(adata.obs["Tissue"].value_counts()))

In [ ]:
# --- run ASTRID ----------------------------------------------------------
# Same four stages as the CLI baseline (--all = clustering + annotation +
# validation + damage). The vignette only runs the first three, so it produces
# no CNV columns.
#
# --out_dir is REQUIRED: without it ASTRID_v0.01.py:647 concatenates
# os.getcwd() + prefix with no separator and writes to a sibling of the repo.
cmd = [
    sys.executable, "ASTRID_v0.01.py", "--all",
    "--input_file",                str(adata_file),
    "--input_prefix",              prefix,
    "--output_file",               str(output_file),
    "--output_clustering_results", str(clustering_result),
    "--out_dir",                   str(outdir),
    "--author_type",               author_type,
]
print(" ".join(cmd), "\n")

# subprocess (not os.system) so the output lands in this cell, not in the
# terminal that launched the kernel. The anndata FutureWarnings are filtered
# from the display but still written to run.log.
t0 = time.time()
proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
with open(log_file, "w") as log:
    for line in proc.stdout:
        log.write(line)
        if "FutureWarning" in line or "warnings.warn" in line:
            continue
        print(line, end="")
rc = proc.wait()
print(f"\nexit {rc} in {(time.time()-t0)/60:.1f} min  ·  full log: {log_file}")

In [ ]:
# --- per-cell view -------------------------------------------------------
out = sc.read_h5ad(output_file)
final_key = out.uns["final_clustering_level"]
print("final clustering level:", final_key,
      "| clusters:", out.obs[final_key].nunique())

plt.rcParams["figure.figsize"] = (6, 6)
sc.pl.umap(out, color=[author_type, "SingleR_CellType"], legend_loc="on data")

In [ ]:
# --- per-cluster view: the result table ----------------------------------
res = pd.read_csv(clustering_result)
res.head()

In [ ]:
# --- first-pass readout --------------------------------------------------
r = subprocess.run([sys.executable, "adrian_spec/read_results.py", str(clustering_result)],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)

In [ ]:
# --- compare against the CLI baseline ------------------------------------
old = pd.read_csv(baseline / "p2_ASTRID_Result.csv")
new = pd.read_csv(clustering_result)
key = new.columns[0]

print(f"clusters  baseline {len(old)}  ·  notebook {len(new)}")
comp = (old[[key, "SingleR_CellType", "CellCount"]]
        .merge(new[[key, "SingleR_CellType", "CellCount"]], on=key,
               how="outer", suffixes=("_base", "_nb")))
same = (comp.SingleR_CellType_base == comp.SingleR_CellType_nb).sum()
print(f"same cluster id + same call: {same}/{len(comp)}")
comp[comp.SingleR_CellType_base != comp.SingleR_CellType_nb].head(20)

### Note on reproducibility

Leiden clustering is seeded, but cluster **ids** can still shift between runs, so a
row-by-row diff may show differences even when the biology is identical. Compare the
composition table from the readout rather than individual cluster labels.

To rebuild p2 from the raw GSE127465 matrix instead of reusing the existing
`p2_preASTRID.h5ad`, follow cells 1–7 of `ASTRID_Vignette.ipynb` but replace the
`used_in_NSCLC_immune` filter with `adata.obs["Patient"] == "p2"` only — otherwise
you get 345 cells instead of 5,166.
